In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# 2. Пути к файлам
base_path = "/content/drive/MyDrive/CelebA"

landmarks_path = f"{base_path}/list_landmarks_align_celeba.txt"
identity_path  = f"{base_path}/identity_CelebA.txt"
partition_path = f"{base_path}/list_eval_partition.txt"

# 3. Загружаем landmarks и добавляем колонку 'filename'
landmarks_df = pd.read_csv(landmarks_path, sep='\s+', skiprows=1)
filenames = [(str(i + 1).zfill(6) + ".jpg") for i in range(len(landmarks_df))]
landmarks_df.insert(0, "filename", filenames)

# 4. Загружаем identity и partition
identity_df  = pd.read_csv(identity_path, sep='\s+', header=None, names=["filename", "identity"])
partition_df = pd.read_csv(partition_path, sep='\s+', header=None, names=["filename", "partition"])

# 5. Объединяем всё в один датафрейм
df = identity_df.merge(landmarks_df, on="filename").merge(partition_df, on="filename")

print(df.head())


In [ ]:
df_train = df[df["partition"] == 0].reset_index(drop=True)
df_val   = df[df["partition"] == 1].reset_index(drop=True)
df_test  = df[df["partition"] == 2].reset_index(drop=True)

print(f"Train: {len(df_train)}")
print(f"Val:   {len(df_val)}")
print(f"Test:  {len(df_test)}")

In [ ]:
# Распакуем во временную папку
!unzip -q /content/drive/MyDrive/CelebA/img_align_celeba.zip -d /content/img_celeba


In [ ]:
img_dir = "/content/img_celeba/img_align_celeba"

In [ ]:
!unzip -q /content/aligned_classifier_train_500.zip -d /content/aligned_classifier_train_500

In [ ]:
train_dir = "/content/aligned_classifier_train_500"

In [ ]:
!unzip -q /content/aligned_classifier_val_500.zip -d /content/aligned_classifier_val_500

In [ ]:
val_dir = "/content/aligned_classifier_val_500"

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def create_heatmap(size, landmark, sigma=3):
    x, y = landmark
    h, w = size
    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    heatmap = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))
    return heatmap

def landmarks_to_heatmaps(image_shape, landmarks, sigma=3):
    return np.stack([
        create_heatmap(image_shape, (x, y), sigma=sigma)
        for (x, y) in landmarks
    ])

def visualize_faces_with_landmarks_and_heatmaps_final(
    img_dir, df, image_ids=None, num_images=6, cols=3, sigma=3, alpha=0.4, cmap='jet'
):
    if image_ids is None:
        image_ids = df['filename'].iloc[:num_images].tolist()

    rows = (len(image_ids) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()

    for idx, image_id in enumerate(image_ids):
        ax = axes[idx]
        img_path = os.path.join(img_dir, image_id)

        img = cv2.imread(img_path)
        if img is None:
            ax.axis('off')
            print(f"Не удалось загрузить {image_id}")
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        row_lm = df[df['filename'] == image_id].iloc[0]

        points = [
            (row_lm['lefteye_x'], row_lm['lefteye_y']),
            (row_lm['righteye_x'], row_lm['righteye_y']),
            (row_lm['nose_x'], row_lm['nose_y']),
            (row_lm['leftmouth_x'], row_lm['leftmouth_y']),
            (row_lm['rightmouth_x'], row_lm['rightmouth_y']),
        ]

        heatmaps = landmarks_to_heatmaps((h, w), points, sigma=sigma)
        combined_heatmap = np.clip(np.sum(heatmaps, axis=0), 0, 1)

        ax.imshow(img)
        ax.imshow(combined_heatmap, cmap=cmap, alpha=alpha)
        ax.set_title(image_id)
        ax.set_xlim([0, w])
        ax.set_ylim([h, 0])
        ax.axis('off')

        for (px, py) in points:
            ax.plot(px, py, 'ro')

    for i in range(len(image_ids), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_faces_with_landmarks_and_heatmaps_final(img_dir, df_train, num_images=6)

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import os

class CelebAHeatmapDataset(Dataset):
    def __init__(self, img_dir, landmarks_df, image_size=256, heatmap_size=64, sigma=2):
        self.img_dir = img_dir
        self.landmarks_df = landmarks_df
        self.image_size = image_size
        self.heatmap_size = heatmap_size
        self.sigma = sigma

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.landmarks_df)

    def __getitem__(self, idx):
        row = self.landmarks_df.iloc[idx]
        fname = row['filename'].strip()
        img_path = os.path.join(self.img_dir, fname)

        image = Image.open(img_path).convert('RGB')
        orig_w, orig_h = image.size
        image = self.transform(image)

        # масштабируем координаты к image_size
        keypoints = np.array([
            [row['lefteye_x'], row['lefteye_y']],
            [row['righteye_x'], row['righteye_y']],
            [row['nose_x'], row['nose_y']],
            [row['leftmouth_x'], row['leftmouth_y']],
            [row['rightmouth_x'], row['rightmouth_y']]
        ], dtype=np.float32)
        scale_x = self.image_size / orig_w
        scale_y = self.image_size / orig_h
        keypoints *= [scale_x, scale_y]

        # создаём heatmap'ы
        heatmaps = self.generate_heatmaps(keypoints)

        return image, torch.tensor(heatmaps, dtype=torch.float32)

    def generate_heatmaps(self, keypoints):
        """Генерация 5 heatmap'ов (один на каждую точку)"""
        hms = np.zeros((len(keypoints), self.heatmap_size, self.heatmap_size), dtype=np.float32)
        for i, (x, y) in enumerate(keypoints):
            if x < 0 or y < 0:
                continue
            x = x * self.heatmap_size / self.image_size
            y = y * self.heatmap_size / self.image_size
            xx, yy = np.meshgrid(np.arange(self.heatmap_size), np.arange(self.heatmap_size))
            hms[i] = np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * self.sigma**2))
        return hms


In [ ]:
train_df_small = df_train.iloc[:7500].reset_index(drop=True)
val_df_small   = df_val.iloc[:1500].reset_index(drop=True)

In [ ]:
train_dataset = CelebAHeatmapDataset(
    img_dir,
    landmarks_df=train_df_small,
    image_size=256,
    heatmap_size=64,
    sigma=2
)
val_dataset = CelebAHeatmapDataset(
    img_dir,
    landmarks_df=val_df_small,
    image_size=256,
    heatmap_size=64,
    sigma=2
)

In [ ]:
len(val_dataset)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch

def visualize_batch_from_dataset(dataset, n=6, k=3, alpha=0.4, cmap='jet'):
    """
    Визуализирует n изображений из датасета с наложенными heatmaps.

    Args:
        dataset: объект torch Dataset
        n (int): сколько примеров показать
        k (int): сколько изображений в строке
        alpha (float): прозрачность heatmap
        cmap (str): colormap для heatmap
    """
    rows = (n + k - 1) // k
    fig, axes = plt.subplots(rows, k, figsize=(k * 4, rows * 4))
    axes = axes.flatten()

    for i in range(n):
        img, heatmaps = dataset[i]

        image_np = img.permute(1, 2, 0).numpy()
        heatmaps_np = heatmaps.numpy()

        # объединение heatmaps и ресайз до размера изображения
        combined = np.clip(np.sum(heatmaps_np, axis=0), 0, 1)
        h_img, w_img = image_np.shape[:2]
        combined_resized = np.array(Image.fromarray((combined * 255).astype(np.uint8)).resize((w_img, h_img)))
        combined_resized = combined_resized / 255.0

        ax = axes[i]
        ax.imshow(image_np)
        ax.imshow(combined_resized, cmap=cmap, alpha=alpha)
        ax.set_title(f"Sample #{i}")
        ax.axis('off')

    # Отключим лишние пустые ячейки
    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_batch_from_dataset(val_dataset, n=16, k=4)

In [ ]:
import pandas
import numpy as np
import matplotlib.pyplot as plt
import seaborn
import torch
import cv2
import os
from time import time
from torch import nn
from tqdm import tqdm
from torch.utils.data import Dataset
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid_channels = out_channels // 2

        self.skip = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1)

        self.conv1 = nn.Conv2d(in_channels, mid_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)

        self.conv2 = nn.Conv2d(mid_channels, mid_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)

        self.conv3 = nn.Conv2d(mid_channels, out_channels, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        skip = self.skip(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        return self.relu(x + skip)

class Hourglass(nn.Module):
    def __init__(self, depth, channels):
        super().__init__()
        self.depth = depth
        self.channels = channels

        self.skip = Residual(channels, channels)
        self.down = nn.Sequential(
            nn.MaxPool2d(2),
            Residual(channels, channels)
        )

        if depth > 1:
            self.inner = Hourglass(depth - 1, channels)
        else:
            self.inner = Residual(channels, channels)

        self.up = Residual(channels, channels)

    def forward(self, x):
        skip = self.skip(x)
        x = self.down(x)
        x = self.inner(x)
        x = self.up(F.interpolate(x, scale_factor=2, mode='nearest'))
        return x + skip

class HourglassBlock(nn.Module):
    def __init__(self, depth, channels, num_keypoints):
        super().__init__()
        self.hourglass = Hourglass(depth, channels)
        self.res = Residual(channels, channels)
        self.out_heatmap = nn.Conv2d(channels, num_keypoints, kernel_size=1)

        # intermediate supervision:
        self.out_feature = nn.Conv2d(channels, channels, kernel_size=1)
        self.out_heatmap_to_feat = nn.Conv2d(num_keypoints, channels, kernel_size=1)

    def forward(self, x):
        hg = self.hourglass(x)
        feat = self.res(hg)
        heatmap = self.out_heatmap(feat)
        feat_next = x + self.out_feature(feat) + self.out_heatmap_to_feat(heatmap)
        return heatmap, feat_next

class StackedHourglassNet(nn.Module):
    def __init__(self, num_stacks, num_blocks, channels, num_keypoints):
        super().__init__()
        self.pre = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            Residual(64, 128),
            nn.MaxPool2d(2),
            Residual(128, 128),
            Residual(128, channels)
        )

        self.stacks = nn.ModuleList([
            HourglassBlock(num_blocks, channels, num_keypoints)
            for _ in range(num_stacks)
        ])

    def forward(self, x):
        x = self.pre(x)
        outputs = []

        for stack in self.stacks:
            heatmap, x = stack(x)
            outputs.append(heatmap)

        return outputs  # список [B, num_keypoints, 64, 64]

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
hourglass = StackedHourglassNet(num_stacks=2, num_blocks=4, channels=128, num_keypoints=5)
hourglass.load_state_dict(torch.load("/content/shg_amazing.pth", map_location='cpu'))
hourglass = hourglass.to(device).eval()

In [ ]:
used_ids = pd.concat([train_df_small, val_df_small])['identity'].unique()

In [ ]:
available_df = df[~df['identity'].isin(used_ids)].copy()

In [ ]:
top_identities = df['identity'].value_counts().head(500).index.tolist()
classifier_df = df[df['identity'].isin(top_identities)].reset_index(drop=True)

print(f"Выбрано {len(top_identities)} identity, всего {len(classifier_df)} изображений")
print("Фото на одного человека (min / max):",
      classifier_df['identity'].value_counts().min(),
      "/",
      classifier_df['identity'].value_counts().max())

In [ ]:
from sklearn.model_selection import train_test_split

train_cls_df, val_cls_df = train_test_split(
    classifier_df,
    test_size=0.2,
    stratify=classifier_df['identity'],
    random_state=42
)


In [ ]:
train_cls_df = train_cls_df.reset_index(drop=True)
val_cls_df = val_cls_df.reset_index(drop=True)

train_cls_dataset = CelebAHeatmapDataset(
    img_dir,
    landmarks_df=train_cls_df,
    image_size=256,
    heatmap_size=64,
    sigma=2
)

val_cls_dataset = CelebAHeatmapDataset(
    img_dir,
    landmarks_df=val_cls_df,
    image_size=256,
    heatmap_size=64,
    sigma=2
)

In [ ]:
def show_aligned_faces(folder_path, n=12, cols=4):
    image_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    image_files = image_files[:n]

    rows = (len(image_files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))

    for ax, fname in zip(axes.flatten(), image_files):
        img = Image.open(os.path.join(folder_path, fname))
        ax.imshow(img)
        ax.set_title(fname)
        ax.axis('off')

    # Отключаем лишние оси
    for ax in axes.flatten()[len(image_files):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# Вызов
show_aligned_faces("/content/aligned_classifier_train_500/10014", n=10, cols=5)


In [ ]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset
import random
from collections import defaultdict

class TripletDatasetFromFolder(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform

        self.image_paths = []
        self.labels = []

        self.label_to_indices = defaultdict(list)
        for label_folder in sorted(self.root_dir.iterdir()):
            if not label_folder.is_dir():
                continue
            label = label_folder.name
            for img_path in label_folder.glob("*.*"):
                self.image_paths.append(str(img_path))
                self.labels.append(label)
                self.label_to_indices[label].append(len(self.image_paths) - 1)

        self.labels_set = sorted(set(self.labels))

    def __getitem__(self, index):
        anchor_path = self.image_paths[index]
        anchor_label = self.labels[index]

        # positive
        positive_index = random.choice(self.label_to_indices[anchor_label])
        while positive_index == index:
            positive_index = random.choice(self.label_to_indices[anchor_label])

        # negative
        negative_label = random.choice([l for l in self.labels_set if l != anchor_label])
        negative_index = random.choice(self.label_to_indices[negative_label])

        a_img = Image.open(self.image_paths[index]).convert("RGB")
        p_img = Image.open(self.image_paths[positive_index]).convert("RGB")
        n_img = Image.open(self.image_paths[negative_index]).convert("RGB")

        if self.transform:
            a_img = self.transform(a_img)
            p_img = self.transform(p_img)
            n_img = self.transform(n_img)

        return a_img, p_img, n_img

    def __len__(self):
        return len(self.image_paths)


In [ ]:
import torchvision.transforms as T
from torchvision.transforms import ToPILImage

train_transform = T.Compose([
    T.Resize((112, 112)),
    T.ToTensor()
])

train_triplet_ds = TripletDatasetFromFolder("/content/aligned_classifier_train_500", transform=train_transform)
val_triplet_ds = TripletDatasetFromFolder("/content/aligned_classifier_val_500", transform=train_transform)

train_loader = torch.utils.data.DataLoader(train_triplet_ds, batch_size=32, shuffle=True, num_workers=4)
val_loader = torch.utils.data.DataLoader(val_triplet_ds, batch_size=32, shuffle=False, num_workers=4)


In [ ]:
import torch.nn as nn
import torchvision.models as models
from torch.optim.lr_scheduler import ReduceLROnPlateau

num_classes = len(train_dataset.classes)

model = models.resnet50(pretrained=True)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3,
    threshold=1e-4, verbose=True
)


In [ ]:
import copy

def train_classifier(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    epochs=10, device='cuda',
    save_path='best_checkpoint.pth'
):
    model.to(device)
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_train_loss = train_loss / total
        epoch_train_acc = correct / total
        train_losses.append(epoch_train_loss)
        train_accuracies.append(epoch_train_acc)
        print(f"Train Loss: {epoch_train_loss:.4f}, Accuracy: {epoch_train_acc:.4f}")

        # Валидация
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)
        print(f"Val   Loss: {epoch_val_loss:.4f}, Accuracy: {epoch_val_acc:.4f}")

        scheduler.step(epoch_val_loss)

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f"Model saved at epoch {epoch+1} (val_loss={epoch_val_loss:.4f})")

    # Сохраняем полный чекпойнт
    torch.save({
        'model_state': best_model_wts,
        'optimizer_state': optimizer.state_dict(),
        'val_loss': best_val_loss,
        'epoch': epoch + 1,
    }, save_path)

    # Восстанавливаем модель
    model.load_state_dict(best_model_wts)

    return train_losses, val_losses, train_accuracies, val_accuracies

In [ ]:
train_losses, val_losses, train_accs, val_accs = train_classifier(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    epochs=30, device=device,
    save_path="resnet_ce_best.pth"
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title("CE: Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title("CE: Accuracy per Epoch")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torch
import os
from PIL import Image

def visualize_predictions(model, dataset, class_names, device='cuda', num_images=16):
    model.eval()
    transform = T.Compose([
        T.Resize((112, 112)),
        T.ToTensor()
    ])

    plt.figure(figsize=(12, 12))
    shown = 0
    for i in range(len(dataset)):
        if shown >= num_images:
            break

        img_path, true_label = dataset.samples[i]
        image = Image.open(img_path).convert('RGB')
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            pred_label = torch.argmax(output, dim=1).item()

        plt.subplot(4, 4, shown + 1)
        plt.imshow(image)
        color = 'green' if pred_label == true_label else 'red'
        plt.title(f"GT: {class_names[true_label]}\nPred: {class_names[pred_label]}", color=color, fontsize=8)
        plt.axis('off')
        shown += 1

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_predictions(model, val_dataset, class_names=val_dataset.classes, device=device, num_images=16)


In [ ]:
import torch.nn.functional as F
import math

class ArcFaceLoss(nn.Module):
    def __init__(self, s=64.0, m=0.5, reduction='mean'):
        super().__init__()
        self.s = s
        self.m = m
        self.reduction = reduction
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, logits, labels):
        cosine = logits
        sine = torch.sqrt(1.0 - torch.clamp(cosine ** 2, 0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = F.one_hot(labels, num_classes=cosine.size(1)).type_as(cosine)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return F.cross_entropy(output, labels, reduction=self.reduction)


In [ ]:
backbone = models.resnet50(pretrained=True)
backbone.fc = nn.Sequential(
    nn.Linear(backbone.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3)
)

num_classes = len(train_dataset.classes)
classifier = nn.Linear(256, num_classes, bias=False)

criterion = ArcFaceLoss(s=64.0, m=0.5)
from torch.optim.lr_scheduler import ReduceLROnPlateau

optimizer = torch.optim.Adam(
    list(backbone.parameters()) + list(classifier.parameters()), lr=1e-4
)

scheduler = ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3,
    threshold=1e-4, verbose=True
)


backbone = backbone.to(device)
classifier = classifier.to(device)


In [ ]:
def train_arcface_classifier(
    model, classifier,
    train_loader, val_loader,
    criterion, optimizer, scheduler,
    epochs=10, device='cuda',
    save_path='best_checkpoint_arcface.pth'
):
    model.to(device)
    classifier.to(device)
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    best_classifier_wts = copy.deepcopy(classifier.state_dict())

    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    for epoch in range(epochs):
        model.train()
        classifier.train()
        train_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            features = model(images)                        # [B, 256]
            features = F.normalize(features)                # L2 norm
            weights = F.normalize(classifier.weight, dim=1) # [C, 256]
            logits = features @ weights.T                   # [B, C]

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, preds = logits.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_train_loss = train_loss / total
        epoch_train_acc = correct / total
        train_losses.append(epoch_train_loss)
        train_accuracies.append(epoch_train_acc)
        print(f"Train Loss: {epoch_train_loss:.4f}, Accuracy: {epoch_train_acc:.4f}")

        # 🔍 Валидация
        model.eval()
        classifier.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                features = model(images)
                features = F.normalize(features)
                weights = F.normalize(classifier.weight, dim=1)
                logits = features @ weights.T

                loss = criterion(logits, labels)
                val_loss += loss.item() * images.size(0)
                _, preds = logits.max(1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)
        print(f"Val   Loss: {epoch_val_loss:.4f}, Accuracy: {epoch_val_acc:.4f}")

        scheduler.step(epoch_val_loss)

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            best_classifier_wts = copy.deepcopy(classifier.state_dict())
            print(f"Model saved at epoch {epoch+1} (val_loss={epoch_val_loss:.4f})")

    torch.save({
        'backbone_state': best_model_wts,
        'classifier_state': best_classifier_wts,
        'optimizer_state': optimizer.state_dict(),
        'val_loss': best_val_loss,
        'epoch': epoch + 1,
    }, save_path)

    model.load_state_dict(best_model_wts)
    classifier.load_state_dict(best_classifier_wts)

    return train_losses, val_losses, train_accuracies, val_accuracies


In [ ]:
train_losses_arc, val_losses_arc, train_accs_arc, val_accs_arc = train_arcface_classifier(
    model=backbone,
    classifier=classifier,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=30,
    device=device,
    save_path="resnet_arcface_best.pth"
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses_arc, label='Train Loss')
plt.plot(val_losses_arc, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title("ArcFace: Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(train_accs_arc, label='Train Accuracy')
plt.plot(val_accs_arc, label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title("ArcFace: Accuracy per Epoch")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
def visualize_arcface_predictions(backbone, classifier, dataset, class_names, device='cuda', num_images=16):
    backbone.eval()
    classifier.eval()

    transform = T.Compose([
        T.Resize((112, 112)),
        T.ToTensor()
    ])

    plt.figure(figsize=(12, 12))
    shown = 0
    for i in range(len(dataset)):
        if shown >= num_images:
            break

        img_path, true_label = dataset.samples[i]
        image = Image.open(img_path).convert('RGB')
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            features = backbone(input_tensor)
            features = F.normalize(features, dim=1)
            weights = F.normalize(classifier.weight, dim=1)
            logits = features @ weights.T
            pred_label = torch.argmax(logits, dim=1).item()

        plt.subplot(4, 4, shown + 1)
        plt.imshow(image)
        color = 'green' if pred_label == true_label else 'red'
        plt.title(f"GT: {class_names[true_label]}\nPred: {class_names[pred_label]}", color=color, fontsize=8)
        plt.axis('off')
        shown += 1

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_arcface_predictions(backbone, classifier, val_dataset, class_names = val_dataset.classes)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss CE')
plt.plot(val_losses, label='Val Loss CE')
plt.plot(train_losses_arc, label='Train Loss ArcFace')
plt.plot(val_losses_arc, label='Val Loss ArcFace')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title("CE vs ArcFace: Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(train_accs, label='Train Accuracy CE')
plt.plot(val_accs, label='Val Accuracy CE')
plt.plot(train_accs_arc, label='Train Accuracy ArcFace')
plt.plot(val_accs_arc, label='Val Accuracy ArcFace')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title("CE vs ArcFace: Accuracy per Epoch")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
class ArcFaceHybridLoss(nn.Module):
    def __init__(self, s=64.0, m=0.5, ce_weight=0.1, reduction='mean'):
        super().__init__()
        self.s = s
        self.m = m
        self.ce_weight = ce_weight
        self.reduction = reduction
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, cosine, labels):
        sine = torch.sqrt(1.0 - torch.clamp(cosine ** 2, 0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = F.one_hot(labels, num_classes=cosine.size(1)).type_as(cosine)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output_arc = output * self.s
        output_ce = cosine * self.s

        loss_arc = F.cross_entropy(output_arc, labels, reduction=self.reduction)
        loss_ce  = F.cross_entropy(output_ce, labels, reduction=self.reduction)

        return loss_arc + self.ce_weight * loss_ce


In [ ]:
backbone = models.resnet50(pretrained=True)
backbone.fc = nn.Sequential(
    nn.Linear(backbone.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3)
)

num_classes = len(train_dataset.classes)
classifier = nn.Linear(256, num_classes, bias=False)

criterion = ArcFaceHybridLoss(s=64.0, m=0.5, ce_weight=0.2)
from torch.optim.lr_scheduler import ReduceLROnPlateau

optimizer = torch.optim.Adam(
    list(backbone.parameters()) + list(classifier.parameters()), lr=1e-4
)

scheduler = ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3,
    threshold=1e-4, verbose=True
)


backbone = backbone.to(device)
classifier = classifier.to(device)


In [ ]:
train_losses_arce, val_losses_arce, train_accs_arce, val_accs_arce = train_arcface_classifier(
    model=backbone,
    classifier=classifier,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=30,
    device=device,
    save_path="resnet_hybrid.pth"
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses_arce, label='Train Loss')
plt.plot(val_losses_arce, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title("Hybrid: Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(train_accs_arce, label='Train Accuracy')
plt.plot(val_accs_arce, label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title("Hybrid: Accuracy per Epoch")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import torch.nn as nn
import torchvision.models as models
from torch.optim.lr_scheduler import ReduceLROnPlateau

class EmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        base_model = models.resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(base_model.children())[:-1])
        self.embedding = nn.Linear(base_model.fc.in_features, embedding_dim)

    def forward(self, x):
        x = self.backbone(x).squeeze()
        x = self.embedding(x)
        return F.normalize(x, p=2, dim=1)

In [ ]:
import torch.nn.functional as F
from torch import nn, optim

def compute_triplet_accuracy(anchor, positive, negative, margin=0.2):
    d_ap = torch.norm(anchor - positive, dim=1)
    d_an = torch.norm(anchor - negative, dim=1)
    return ((d_ap + margin) < d_an).float().mean().item()


In [ ]:
def train_triplet_model(
    model,
    train_loader, val_loader,
    criterion, optimizer, scheduler,
    epochs=10, device='cuda',
    save_path='best_checkpoint_triplet.pth'
):
    import copy
    model.to(device)
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    for epoch in range(epochs):
        model.train()
        train_loss, train_acc, total_train = 0.0, 0.0, 0

        for a, p, n in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            a, p, n = a.to(device), p.to(device), n.to(device)
            batch_size = a.size(0)

            anchor = model(a)
            positive = model(p)
            negative = model(n)

            loss = criterion(anchor, positive, negative)
            acc = compute_triplet_accuracy(anchor, positive, negative)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_size
            train_acc += acc * batch_size
            total_train += batch_size

        epoch_train_loss = train_loss / total_train
        epoch_train_acc = train_acc / total_train
        train_losses.append(epoch_train_loss)
        train_accuracies.append(epoch_train_acc)
        print(f"Train Loss: {epoch_train_loss:.4f}, Triplet Train Acc: {epoch_train_acc:.4f}")

        model.eval()
        val_loss, val_acc, total_val = 0.0, 0.0, 0
        with torch.no_grad():
            for a, p, n in val_loader:
                a, p, n = a.to(device), p.to(device), n.to(device)
                batch_size = a.size(0)

                anchor = model(a)
                positive = model(p)
                negative = model(n)

                loss = criterion(anchor, positive, negative)
                acc = compute_triplet_accuracy(anchor, positive, negative)

                val_loss += loss.item() * batch_size
                val_acc += acc * batch_size
                total_val += batch_size

        epoch_val_loss = val_loss / total_val
        epoch_val_acc = val_acc / total_val
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)
        print(f"Val Loss: {epoch_val_loss:.4f}, Triplet Val Acc: {epoch_val_acc:.4f}")

        scheduler.step(epoch_val_loss)

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f"Model saved at epoch {epoch+1} (val_loss={epoch_val_loss:.4f})")

    torch.save({
        'model_state': best_model_wts,
        'optimizer_state': optimizer.state_dict(),
        'val_loss': best_val_loss,
        'epoch': epoch + 1,
    }, save_path)

    model.load_state_dict(best_model_wts)

    return train_losses, val_losses, train_accuracies, val_accuracies


In [ ]:
model = EmbeddingNet(embedding_dim=256).to(device)
criterion = nn.TripletMarginLoss(margin=0.3, p=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

train_losses_tri, val_losses_tri, train_accs_tri, val_accs_tri = train_triplet_model(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=30,
    device=device,
    save_path="best_triplet_model.pth"
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses_tri, label='Train Loss')
plt.plot(val_losses_tri, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title("Triplet: Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(train_accs_tri, label='Train Accuracy')
plt.plot(val_accs_tri, label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title("Triplet: Accuracy per Epoch")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
torch.save(model.state_dict(), "triplet_final_margin03_valacc85.pth")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss CE')
plt.plot(val_losses, label='Val Loss CE')
plt.plot(train_losses_arc, label='Train Loss ArcFace')
plt.plot(val_losses_arc, label='Val Loss ArcFace')
plt.plot(train_losses_arce, label='Train Loss Hybrid')
plt.plot(val_losses_arce, label='Val Loss Hybrid')
plt.plot(train_losses_tri, label='Train Loss Triplet')
plt.plot(val_losses_tri, label='Val Loss Triplet')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title("CE vs ArcFace: Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(train_accs, label='Train Accuracy CE')
plt.plot(val_accs, label='Val Accuracy CE')
plt.plot(train_accs_arc, label='Train Accuracy ArcFace')
plt.plot(val_accs_arc, label='Val Accuracy ArcFace')
plt.plot(train_accs_arce, label='Train Accuracy Hybrid')
plt.plot(val_accs_arce, label='Val Accuracy Hybrid')
plt.plot(train_accs_tri, label='Train Accuracy Triplet')
plt.plot(val_accs_tri, label='Val Accuracy Triplet')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title("CE vs ArcFace: Accuracy per Epoch")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Создаём модель CE
ce_model = models.resnet50(pretrained=False)
ce_model.fc = nn.Sequential(
    nn.Linear(ce_model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)  # финальный слой с bias
)
ce_model.load_state_dict(torch.load("resnet_ce_best.pth")['model_state'])
ce_model = ce_model.to(device).eval()


In [ ]:
# Создаём модель ArcFace
arc_backbone = models.resnet50(pretrained=False)
arc_backbone.fc = nn.Sequential(
    nn.Linear(arc_backbone.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3)
)
arc_classifier = nn.Linear(256, num_classes, bias=False)

checkpoint = torch.load("resnet_arcface_best.pth")
arc_backbone.load_state_dict(checkpoint['backbone_state'])
arc_classifier.load_state_dict(checkpoint['classifier_state'])

arc_backbone = arc_backbone.to(device).eval()
arc_classifier = arc_classifier.to(device).eval()


In [ ]:
# Создаём модель Hybrid
hybrid_backbone = models.resnet50(pretrained=False)
hybrid_backbone.fc = nn.Sequential(
    nn.Linear(hybrid_backbone.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3)
)
hybrid_classifier = nn.Linear(256, num_classes, bias=False)

checkpoint = torch.load("resnet_hybrid.pth")
hybrid_backbone.load_state_dict(checkpoint['backbone_state'])
hybrid_classifier.load_state_dict(checkpoint['classifier_state'])

hybrid_backbone = hybrid_backbone.to(device).eval()
hybrid_classifier = hybrid_classifier.to(device).eval()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from collections import Counter

def plot_confusion_matrix(
    model=None,
    backbone=None,
    classifier=None,
    dataloader=None,
    class_names=None,
    device='cuda',
    max_classes=20,
    mode='ce'  # one of: 'ce', 'arcface', 'hybrid'
):
    assert mode in ['ce', 'arcface', 'hybrid'], "mode must be one of ['ce', 'arcface', 'hybrid']"

    y_true = []
    y_pred = []

    if mode == 'ce':
        assert model is not None, "You must pass `model` for CE mode"
        model.eval()
    else:
        assert backbone is not None and classifier is not None, "You must pass `backbone` and `classifier` for ArcFace/Hybrid"
        backbone.eval()
        classifier.eval()

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Evaluating ({mode})"):
            images = images.to(device)
            labels = labels.to(device)

            if mode == 'ce':
                logits = model(images)
            else:
                feats = F.normalize(backbone(images), dim=1)
                weights = F.normalize(classifier.weight, dim=1)
                logits = feats @ weights.T

            preds = torch.argmax(logits, dim=1)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    most_common = [cls for cls, _ in Counter(y_true).most_common(max_classes)]
    mask = np.isin(y_true, most_common) & np.isin(y_pred, most_common)
    y_true_filtered = y_true[mask]
    y_pred_filtered = y_pred[mask]
    top_labels = sorted(set(y_true_filtered) | set(y_pred_filtered))

    cm = confusion_matrix(y_true_filtered, y_pred_filtered, labels=top_labels)
    labels_display = [class_names[i] for i in top_labels]

    acc = (y_true_filtered == y_pred_filtered).sum() / len(y_true_filtered)

    fig, ax = plt.subplots(figsize=(10, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_display)
    disp.plot(include_values=True, cmap='Blues', ax=ax, xticks_rotation='vertical')
    plt.title(f"Confusion Matrix ({mode.upper()}) — Acc: {acc:.3f}")
    plt.tight_layout()
    plt.show()


In [ ]:
# Для CrossEntropy модели
plot_confusion_matrix(
    model=ce_model,
    dataloader=val_loader,
    class_names=val_dataset.classes,
    device=device,
    mode='ce'
)

# Для ArcFace
plot_confusion_matrix(
    backbone=arc_backbone,
    classifier=arc_classifier,
    dataloader=val_loader,
    class_names=val_dataset.classes,
    device=device,
    mode='arcface'
)

# Для гибридной модели
plot_confusion_matrix(
    backbone=hybrid_backbone,
    classifier=hybrid_classifier,
    dataloader=val_loader,
    class_names=val_dataset.classes,
    device=device,
    mode='hybrid'
)


In [ ]:
def evaluate_topk(
    backbone, dataloader,
    classifier=None,
    device='cuda',
    mode='ce',  # 'ce', 'arcface', 'hybrid'
    k=(1, 5)
):
    backbone.eval()
    if classifier is not None:
        classifier.eval()

    correct_top1 = 0
    correct_top5 = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating Top-k"):
            images = images.to(device)
            labels = labels.to(device)

            if mode == 'ce':
                logits = backbone(images)

            elif mode in ['arcface', 'hybrid']:
                features = F.normalize(backbone(images), dim=1)
                weights = F.normalize(classifier.weight, dim=1)
                logits = features @ weights.T

            else:
                raise ValueError(f"Unknown mode: {mode}")

            _, topk_preds = logits.topk(max(k), dim=1, largest=True, sorted=True)
            correct = topk_preds.eq(labels.view(-1, 1).expand_as(topk_preds))

            correct_top1 += correct[:, 0].sum().item()
            correct_top5 += correct.any(dim=1).sum().item()
            total += labels.size(0)

    acc1 = correct_top1 / total
    acc5 = correct_top5 / total
    print()
    print(mode.upper())
    print(f"Top-1 Accuracy: {acc1:.4f}")
    print(f"Top-5 Accuracy: {acc5:.4f}")


In [ ]:
# CrossEntropy
evaluate_topk(ce_model, val_loader, device=device, mode='ce')

# ArcFace
evaluate_topk(arc_backbone, val_loader, classifier=arc_classifier, device=device, mode='arcface')

# Hybrid
evaluate_topk(hybrid_backbone, val_loader, classifier=hybrid_classifier, device=device, mode='hybrid')


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def compute_tsne(backbone, classifier=None, dataloader=None, mode='CE', max_classes=30, per_class=30):
    backbone.eval()
    if classifier: classifier.eval()

    feats, labels = [], []
    counts = {}

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Extracting"):
            x, y = x.to(device), y.to(device)
            f = backbone(x)
            f = F.normalize(f)

            for feat, label in zip(f, y):
                label = label.item()
                if label not in counts:
                    counts[label] = 0
                if counts[label] < per_class:
                    feats.append(feat.cpu().numpy())
                    labels.append(label)
                    counts[label] += 1
            if len(counts) >= max_classes and all(c >= per_class for c in counts.values()):
                break

    feats = np.array(feats)
    labels = np.array(labels)

    tsne = TSNE(n_components=2, init='pca', random_state=42, perplexity=30)
    reduced = tsne.fit_transform(feats)

    plt.figure(figsize=(8, 8))
    plt.scatter(reduced[:, 0], reduced[:, 1], c=labels, cmap='tab20', s=10)
    plt.title(f"t-SNE {mode}")
    plt.axis('off')
    plt.show()


In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as T

val_tsne_transform = T.Compose([
    T.Resize((112, 112)),
    T.ToTensor()
])

val_tsne_dataset = ImageFolder("/content/aligned_classifier_val_500", transform=val_tsne_transform)
val_tsne_loader = DataLoader(val_tsne_dataset, batch_size=64, shuffle=True, num_workers=2)


In [ ]:
compute_tsne(model, classifier=None, dataloader=val_tsne_loader, mode = 'Triplet')

In [ ]:
compute_tsne(ce_model, dataloader=val_loader, mode = 'CE')  # CE
compute_tsne(arc_backbone, classifier=arc_classifier, dataloader=val_loader, mode = 'ArcFace')  # ArcFace
compute_tsne(hybrid_backbone, classifier=hybrid_classifier, dataloader=val_loader, mode = 'Hybrid')  # Hybrid
